<div align="center">
<h1><b>Data Cleaning and Preprocessing</b></h1>
</div>

In [1]:
# Importing the required Python Libraries
import json
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Load time index
time_df = pd.read_csv('../data/raw/time.csv')

# Build proper datetime column
time_df['timestamp'] = pd.to_datetime({
    'year':   time_df['year'],
    'month':  time_df['month'],
    'day':    time_df['day'],
    'hour':   time_df['hour'],
    'minute': time_df['minute'],
    'second': time_df['second']
})

# Add a slot index (1 to 8640) to match other files
time_df['slot'] = range(1, len(time_df) + 1)

print("Time range:", time_df['timestamp'].min(), "→", time_df['timestamp'].max())
print("Total slots:", len(time_df))
print(time_df[['slot','timestamp']].head())

Time range: 2022-06-19 00:00:00 → 2022-07-18 23:55:00
Total slots: 8640
   slot           timestamp
0     1 2022-06-19 00:00:00
1     2 2022-06-19 00:05:00
2     3 2022-06-19 00:10:00
3     4 2022-06-19 00:15:00
4     5 2022-06-19 00:20:00


In [3]:
# Load all 8640×248 files
occ_df  = pd.read_csv('../data/raw/occupancy.csv')
vol_df  = pd.read_csv('../data/raw/volume.csv')
pri_df  = pd.read_csv('../data/raw/price.csv')
dur_df  = pd.read_csv('../data/raw/duration.csv')

print("Occupancy:", occ_df.shape)
print("Volume:   ", vol_df.shape)
print("Price:    ", pri_df.shape)
print("Duration: ", dur_df.shape)

# Confirm timestamp column is integer index
print("\nTimestamp sample:", occ_df['timestamp'].head().tolist())
occ_df.head()

Occupancy: (8640, 248)
Volume:    (8640, 248)
Price:     (8640, 248)
Duration:  (8640, 248)

Timestamp sample: [1, 2, 3, 4, 5]


,timestamp,102,105,107,108,109,110,111,115,123,...,1160,1162,1163,1164,1166,1167,1168,1170,1172,1173
0,1,12,16,24,15,6,8,24,1,2,...,0,12,1,38,26,162,10,1,8,15
1,2,12,16,24,15,6,8,24,1,2,...,0,12,1,38,26,162,10,1,8,15
2,3,12,16,24,15,6,8,24,1,2,...,0,12,1,38,26,164,10,1,8,15
3,4,12,16,24,15,6,8,24,1,2,...,0,12,1,38,26,166,10,1,8,15
4,5,12,16,24,15,6,8,24,1,2,...,0,12,1,38,26,168,10,1,8,15


In [4]:
def melt_wide_to_long(df, value_name):
    """Convert wide station-time matrix to long format"""
    melted = df.melt(
        id_vars='timestamp',
        var_name='station_id',
        value_name=value_name
    )
    melted['station_id'] = melted['station_id'].astype(int)
    melted['timestamp']  = melted['timestamp'].astype(int)
    return melted

occ_long  = melt_wide_to_long(occ_df,  'occupancy')
vol_long  = melt_wide_to_long(vol_df,  'volume')
pri_long  = melt_wide_to_long(pri_df,  'price_multiplier')
dur_long  = melt_wide_to_long(dur_df,  'avg_duration_hrs')

print("Occupancy long shape:", occ_long.shape)
print("Each file should have:", 8640 * 247, "rows")
print(occ_long.head())

Occupancy long shape: (2134080, 3)
Each file should have: 2134080 rows
   timestamp  station_id  occupancy
0          1         102         12
1          2         102         12
2          3         102         12
3          4         102         12
4          5         102         12


In [5]:
# Merging the 4 datasets into one
# Start with occupancy as base
urban_df = occ_long.copy()

# Merge volume
urban_df = urban_df.merge(vol_long,  on=['timestamp','station_id'], how='left')

# Merge price multiplier
urban_df = urban_df.merge(pri_long,  on=['timestamp','station_id'], how='left')

# Merge duration
urban_df = urban_df.merge(dur_long,  on=['timestamp','station_id'], how='left')

print("Merged urban shape:", urban_df.shape)
print(urban_df.head())

Merged urban shape: (2134080, 6)
   timestamp  station_id  occupancy    volume  price_multiplier  \
0          1         102         12  2.858333             0.924   
1          2         102         12  4.375000             0.924   
2          3         102         12  4.375000             0.924   
3          4         102         12  4.375000             0.924   
4          5         102         12  4.375000             0.924   

   avg_duration_hrs  
0              0.49  
1              0.75  
2              0.75  
3              0.75  
4              0.75  


In [6]:
# Merge actual timestamps using slot index
urban_df = urban_df.merge(
    time_df[['slot','timestamp']].rename(columns={  
        'timestamp': 'datetime',
        'slot': 'timestamp'
    }),
    on='timestamp',
    how='left'
)

# Now rename for clarity
urban_df = urban_df.rename(columns={'timestamp': 'slot_index'})

print("Columns now:", urban_df.columns.tolist())
print(urban_df.head())

Columns now: ['slot_index', 'station_id', 'occupancy', 'volume', 'price_multiplier', 'avg_duration_hrs', 'datetime']
   slot_index  station_id  occupancy    volume  price_multiplier  \
0           1         102         12  2.858333             0.924   
1           2         102         12  4.375000             0.924   
2           3         102         12  4.375000             0.924   
3           4         102         12  4.375000             0.924   
4           5         102         12  4.375000             0.924   

   avg_duration_hrs            datetime  
0              0.49 2022-06-19 00:00:00  
1              0.75 2022-06-19 00:05:00  
2              0.75 2022-06-19 00:10:00  
3              0.75 2022-06-19 00:15:00  
4              0.75 2022-06-19 00:20:00  


In [7]:
# Add Station Metadata into the urban_df
info_df = pd.read_csv('../data/raw/information.csv')

# 'grid' in information.csv = station_id in our long format
info_df = info_df.rename(columns={
    'grid':          'station_id',
    'lon':           'longitude',
    'la':            'latitude',
    'count':         'total_chargers',
    'fast_count':    'fast_chargers',
    'slow_count':    'slow_chargers',
    'area':          'area_sqkm',
    'CBD':           'is_cbd',
    'dynamic_pricing': 'has_dynamic_pricing'
})

# Keep relevant columns
info_cols = ['station_id','longitude','latitude','total_chargers',
             'fast_chargers','slow_chargers','area_sqkm','is_cbd','has_dynamic_pricing']

urban_df = urban_df.merge(info_df[info_cols], on='station_id', how='left')

print("After metadata merge:", urban_df.shape)
print(urban_df.head())

After metadata merge: (2134080, 15)
   slot_index  station_id  occupancy    volume  price_multiplier  \
0           1         102         12  2.858333             0.924   
1           2         102         12  4.375000             0.924   
2           3         102         12  4.375000             0.924   
3           4         102         12  4.375000             0.924   
4           5         102         12  4.375000             0.924   

   avg_duration_hrs            datetime  longitude  latitude  total_chargers  \
0              0.49 2022-06-19 00:00:00    114.103  22.54041              30   
1              0.75 2022-06-19 00:05:00    114.103  22.54041              30   
2              0.75 2022-06-19 00:10:00    114.103  22.54041              30   
3              0.75 2022-06-19 00:15:00    114.103  22.54041              30   
4              0.75 2022-06-19 00:20:00    114.103  22.54041              30   

   fast_chargers  slow_chargers  area_sqkm  is_cbd  has_dynamic_pricing  


In [8]:
# Addition of time features
urban_df['hour']        = urban_df['datetime'].dt.hour
urban_df['minute']      = urban_df['datetime'].dt.minute
urban_df['day_of_week'] = urban_df['datetime'].dt.dayofweek   # 0=Monday
urban_df['day_name']    = urban_df['datetime'].dt.day_name()
urban_df['date']        = urban_df['datetime'].dt.date
urban_df['is_weekend']  = urban_df['day_of_week'].isin([5,6]).astype(int)
urban_df['month']       = urban_df['datetime'].dt.month
urban_df['week']        = urban_df['datetime'].dt.isocalendar().week.astype(int)

# Period labeling
def label_period(hour):
    if 7 <= hour <= 9 or 17 <= hour <= 20:
        return 'peak'
    elif 10 <= hour <= 16:
        return 'shoulder'
    else:
        return 'off_peak'

urban_df['period'] = urban_df['hour'].apply(label_period)

print("Time features added")
print(urban_df.head())

Time features added
   slot_index  station_id  occupancy    volume  price_multiplier  \
0           1         102         12  2.858333             0.924   
1           2         102         12  4.375000             0.924   
2           3         102         12  4.375000             0.924   
3           4         102         12  4.375000             0.924   
4           5         102         12  4.375000             0.924   

   avg_duration_hrs            datetime  longitude  latitude  total_chargers  \
0              0.49 2022-06-19 00:00:00    114.103  22.54041              30   
1              0.75 2022-06-19 00:05:00    114.103  22.54041              30   
2              0.75 2022-06-19 00:10:00    114.103  22.54041              30   
3              0.75 2022-06-19 00:15:00    114.103  22.54041              30   
4              0.75 2022-06-19 00:20:00    114.103  22.54041              30   

   ...  has_dynamic_pricing  hour  minute  day_of_week  day_name        date  \
0  ...    

In [9]:
# FEATURE ENGINEERING FOR URBANEV DATASET:
BASE_TARIFF_CNY = 1.2  # Base price in CNY/kWh (Shenzhen baseline)

# 1. Charger Utilization Rate = occupancy / total chargers
urban_df['charger_utilization_rate'] = (
    urban_df['occupancy'] / urban_df['total_chargers']
).clip(0, 1)

# 2. Absolute price from multiplier × base tariff
urban_df['price_cny_kwh'] = urban_df['price_multiplier'] * BASE_TARIFF_CNY

# 3. Revenue proxy per slot per station
#    Revenue ≈ volume × avg_duration × price
urban_df['revenue_proxy'] = (
    urban_df['volume'] *
    urban_df['avg_duration_hrs'] *
    urban_df['price_cny_kwh']
)

# 4. Queue Length Proxy — volume when utilization is high
urban_df['queue_proxy'] = np.where(
    urban_df['charger_utilization_rate'] >= 0.8,
    urban_df['volume'] - urban_df['occupancy'],
    0
).clip(0)

# 5. Occupancy Density = occupancy per sq km
urban_df['occupancy_density'] = (
    urban_df['occupancy'] / urban_df['area_sqkm']
)

# 6. Charger type flag
urban_df['has_fast_charger'] = (urban_df['fast_chargers'] > 0).astype(int)

# 7. Congestion flag
urban_df['is_congested'] = (urban_df['charger_utilization_rate'] > 0.8).astype(int)

# 8. Underutilized flag
urban_df['is_underutilized'] = (urban_df['charger_utilization_rate'] < 0.3).astype(int)

print("Feature engineering complete for Urban EV Dataset")
print(urban_df.head())

Feature engineering complete for Urban EV Dataset
   slot_index  station_id  occupancy    volume  price_multiplier  \
0           1         102         12  2.858333             0.924   
1           2         102         12  4.375000             0.924   
2           3         102         12  4.375000             0.924   
3           4         102         12  4.375000             0.924   
4           5         102         12  4.375000             0.924   

   avg_duration_hrs            datetime  longitude  latitude  total_chargers  \
0              0.49 2022-06-19 00:00:00    114.103  22.54041              30   
1              0.75 2022-06-19 00:05:00    114.103  22.54041              30   
2              0.75 2022-06-19 00:10:00    114.103  22.54041              30   
3              0.75 2022-06-19 00:15:00    114.103  22.54041              30   
4              0.75 2022-06-19 00:20:00    114.103  22.54041              30   

   ...  week    period  charger_utilization_rate  price_cny_

In [10]:
# Handling Missinng values for UrbanEV Dataset
print("=== Missing Values Before Handling ===")
print(urban_df.isnull().sum()[urban_df.isnull().sum() > 0])

# Document assumptions
assumptions_urban = {
    'occupancy':    'Zero-filled — missing slot means no chargers occupied',
    'volume':       'Zero-filled — missing slot means no sessions',
    'avg_duration_hrs': 'Forward-fill within station — duration unlikely to change drastically',
    'price_multiplier': 'Fill with 1.0 — neutral multiplier if price data missing',
}

urban_df['occupancy']         = urban_df['occupancy'].fillna(0)
urban_df['volume']            = urban_df['volume'].fillna(0)
urban_df['price_multiplier']  = urban_df['price_multiplier'].fillna(1.0)
urban_df['avg_duration_hrs']  = urban_df.groupby('station_id')['avg_duration_hrs'].ffill().bfill()

# Drop rows where datetime is null (shouldn't happen but safety check)
urban_df = urban_df.dropna(subset=['datetime'])

print("\n=== Missing Values After Handling ===")
print(urban_df.isnull().sum()[urban_df.isnull().sum() > 0])
print("\nFinal urban_df shape:", urban_df.shape)

=== Missing Values Before Handling ===
Series([], dtype: int64)

=== Missing Values After Handling ===
Series([], dtype: int64)

Final urban_df shape: (2134080, 32)


In [11]:
# Loading and Processing the ACN Dataset
with open('../data/raw/acndata_sessions.json', 'r') as f:
    acn_raw = json.load(f)

print("Meta info:")
print(acn_raw['_meta'])
print("\nTotal sessions:", len(acn_raw['_items']))

# Convert to DataFrame
acn_df = pd.DataFrame(acn_raw['_items'])
print("\nColumns:", acn_df.columns.tolist())
print(acn_df.shape)
print(acn_df.head())

Meta info:
{'end': 'Sun, 16 Dec 2018 23:59:00 GMT', 'min_kWh': None, 'site': 'caltech', 'start': 'Wed, 25 Apr 2018 00:00:00 GMT'}

Total sessions: 15013

Columns: ['_id', 'clusterID', 'connectionTime', 'disconnectTime', 'doneChargingTime', 'kWhDelivered', 'sessionID', 'siteID', 'spaceID', 'stationID', 'timezone', 'userID', 'userInputs']
(15013, 13)
                        _id clusterID                 connectionTime  \
0  5bc90cb9f9af8b0d7fe77cd2      0039  Wed, 25 Apr 2018 11:08:04 GMT   
1  5bc90cb9f9af8b0d7fe77cd3      0039  Wed, 25 Apr 2018 13:45:10 GMT   
2  5bc90cb9f9af8b0d7fe77cd4      0039  Wed, 25 Apr 2018 13:45:50 GMT   
3  5bc90cb9f9af8b0d7fe77cd5      0039  Wed, 25 Apr 2018 14:37:06 GMT   
4  5bc90cb9f9af8b0d7fe77cd6      0039  Wed, 25 Apr 2018 14:40:34 GMT   

                  disconnectTime               doneChargingTime  kWhDelivered  \
0  Wed, 25 Apr 2018 13:20:10 GMT  Wed, 25 Apr 2018 13:21:10 GMT         7.932   
1  Thu, 26 Apr 2018 00:56:16 GMT  Wed, 25 Apr 2018 16:

In [12]:
# Parse timestamps
acn_df['connectionTime']    = pd.to_datetime(acn_df['connectionTime'],   utc=True)
acn_df['disconnectTime']    = pd.to_datetime(acn_df['disconnectTime'],   utc=True)
acn_df['doneChargingTime']  = pd.to_datetime(acn_df['doneChargingTime'], utc=True)

# Convert to naive datetime (remove timezone)
acn_df['connectionTime']   = acn_df['connectionTime'].dt.tz_localize(None)
acn_df['disconnectTime']   = acn_df['disconnectTime'].dt.tz_localize(None)
acn_df['doneChargingTime'] = acn_df['doneChargingTime'].dt.tz_localize(None)

# Session duration
acn_df['session_duration_hrs'] = (
    acn_df['disconnectTime'] - acn_df['connectionTime']
).dt.total_seconds() / 3600

# Active charging duration (done - connected)
acn_df['charging_duration_hrs'] = (
    acn_df['doneChargingTime'] - acn_df['connectionTime']
).dt.total_seconds() / 3600

# Drop bad rows
before = len(acn_df)
acn_df = acn_df[acn_df['session_duration_hrs'] > 0]
acn_df = acn_df.dropna(subset=['kWhDelivered','connectionTime','disconnectTime'])
acn_df = acn_df[acn_df['kWhDelivered'] > 0]
print(f"Dropped {before - len(acn_df)} invalid rows. Remaining: {len(acn_df)}")

Dropped 0 invalid rows. Remaining: 15013


In [ ]:
# FEATURE ENGINEERING (ACN DATA)
BASE_TARIFF_INR = 15  # ₹/kWh as per project baseline

# 1. Time features
acn_df['hour']        = acn_df['connectionTime'].dt.hour
acn_df['day_of_week'] = acn_df['connectionTime'].dt.dayofweek
acn_df['is_weekend']  = acn_df['day_of_week'].isin([5,6]).astype(int)
acn_df['month']       = acn_df['connectionTime'].dt.month
acn_df['date']        = acn_df['connectionTime'].dt.date
acn_df['day_name']    = acn_df['connectionTime'].dt.day_name()

# 2. Period label
acn_df['period'] = acn_df['hour'].apply(label_period)

# 3. Revenue per session at baseline
acn_df['revenue_session_inr'] = acn_df['kWhDelivered'] * BASE_TARIFF_INR

# 4. Power rate delivered (kW)
acn_df['avg_power_kw'] = acn_df['kWhDelivered'] / acn_df['session_duration_hrs'].replace(0, np.nan)

# 5. Charger utilization rate per session (charging time / 24hrs)
acn_df['charger_utilization_rate'] = (
    acn_df['session_duration_hrs'] / 24
).clip(0, 1)

# 6. Idle time = total session - actual charging
acn_df['idle_time_hrs'] = (
    acn_df['session_duration_hrs'] - acn_df['charging_duration_hrs']
).clip(0)

# 7. Daily occupancy per station
daily_occ = acn_df.groupby(['stationID','date']).size().reset_index(name='daily_sessions')
acn_df = acn_df.merge(daily_occ, on=['stationID','date'], how='left')
acn_df = acn_df.rename(columns={'daily_sessions': 'occupancy_density'})

# 8. Queue proxy — same station, overlapping sessions
acn_df = acn_df.sort_values(['stationID','connectionTime'])
acn_df['queue_proxy'] = acn_df.groupby(['stationID','date']).cumcount()

print("ACN feature engineering complete")
print(acn_df.head())

ACN feature engineering complete
                        _id clusterID      connectionTime      disconnectTime  \
0  5bc90cb9f9af8b0d7fe77ce9      0039 2018-04-25 16:01:26 2018-04-26 00:33:06   
1  5bc90cb9f9af8b0d7fe77d06      0039 2018-04-26 02:17:10 2018-04-26 04:29:39   
2  5bc91212f9af8b0d98ff68dc      0039 2018-04-26 16:01:58 2018-04-26 19:20:15   
3  5bc9153cf9af8b0dad3c05fb      0039 2018-04-27 16:25:09 2018-04-27 19:19:59   
4  5bc91570f9af8b0dad3c0624      0039 2018-04-28 15:17:54 2018-04-28 16:52:05   

     doneChargingTime  kWhDelivered                               sessionID  \
0 2018-04-26 00:14:16        14.693  2_39_123_23_2018-04-25 16:01:25.950919   
1 2018-04-26 04:29:34        13.892  2_39_123_23_2018-04-26 02:17:10.267290   
2 2018-04-26 19:20:10        10.248  2_39_123_23_2018-04-26 16:01:58.184015   
3 2018-04-27 19:19:55         6.444  2_39_123_23_2018-04-27 16:25:09.116295   
4 2018-04-28 16:51:57         5.275  2_39_123_23_2018-04-28 15:17:54.026514   

  sit

In [ ]:
# Missing Value Handling for ACN Dataset

print("=== ACN Missing Values ===")
print(acn_df.isnull().sum()[acn_df.isnull().sum() > 0])

assumptions_acn = {
    'doneChargingTime': 'Null means charging ran till disconnect — use disconnectTime',
    'userID':           'Nulls retained — anonymous sessions are valid data',
    'clusterID':        'Nulls forward-filled within siteID',
    'avg_power_kw':     'Nulls from zero-duration sessions — already dropped',
}

# Fill doneChargingTime nulls with disconnectTime
acn_df['doneChargingTime'] = acn_df['doneChargingTime'].fillna(acn_df['disconnectTime'])

# Fill clusterID within site
acn_df['clusterID'] = acn_df.groupby('siteID')['clusterID'].ffill().bfill()

print("\n=== ACN Missing Values After ===")
print(acn_df.isnull().sum()[acn_df.isnull().sum() > 0])
print("\nFinal ACN shape:", acn_df.shape)

=== ACN Missing Values ===
doneChargingTime             8
userID                   12770
userInputs               12770
charging_duration_hrs        8
idle_time_hrs                8
dtype: int64

=== ACN Missing Values After ===
userID                   12770
userInputs               12770
charging_duration_hrs        8
idle_time_hrs                8
dtype: int64

Final ACN shape: (15013, 28)


In [17]:
# Exporting all the cleaned files:
# Export Urban EV clean data
urban_df.to_csv('../data/processed/clean_urban_data.csv', index=False)
print("Saved: clean_urban_data.csv →", urban_df.shape)

# Export ACN clean data
acn_df.to_csv('../data/processed/clean_acn_data.csv', index=False)
print("Saved: clean_acn_data.csv →", acn_df.shape)

# Export station info for reference
info_df.to_csv('../data/processed/station_metadata.csv', index=False)
print("Saved: station_metadata.csv →", info_df.shape)

print("\nNotebook 1 Complete")
print(f"   Urban EV: {urban_df.shape[0]:,} rows × {urban_df.shape[1]} columns")
print(f"   ACN Data: {acn_df.shape[0]:,} rows × {acn_df.shape[1]} columns")

Saved: clean_urban_data.csv → (2134080, 32)
Saved: clean_acn_data.csv → (15013, 29)
Saved: station_metadata.csv → (247, 10)

Notebook 1 Complete
   Urban EV: 2,134,080 rows × 32 columns
   ACN Data: 15,013 rows × 29 columns
